In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from OSDE.LegendreExpSPDensity import LegExp, LegExpSPDensity, LegExpResult
from StocProcess.RBM import RBMTransProb, MakeRBMTransProbFunc
from QAE.RQAE import RQAE

In [2]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
mu = 0.5
sigma = 1.0
n_terms = 5

tN = 0.6
t1 = 0.2

# approximation setting
maxDeg = 5
R = 12
eps0 = 1 / 2**10
integEpsabs = 1e-4
nRep = 10

In [3]:
np.random.seed(1)

In [4]:
Ns = np.tile((2 ** np.linspace(3, 6, 7)).astype(int), nRep)
print(Ns)

[ 8 11 16 22 32 45 64  8 11 16 22 32 45 64  8 11 16 22 32 45 64  8 11 16
 22 32 45 64  8 11 16 22 32 45 64  8 11 16 22 32 45 64  8 11 16 22 32 45
 64  8 11 16 22 32 45 64  8 11 16 22 32 45 64  8 11 16 22 32 45 64]


In [5]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(tN, t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [6]:
epss = []
pEsts = []
totalQueryNums = []
maxDepths = []
NsComp = []

for N in Ns:
    legExpResultPrev = None
    ts = np.concatenate([[t0], np.linspace(t1, tN, N)])
    eps = eps0 / np.sqrt(N)
    epss.append(eps)
    totalQueryNum = 0
    maxDepth = 0
    print(datetime.datetime.now(), "N=", N)

    for i in range(len(ts)-1):
        print("i_t=", i, datetime.datetime.now())
        transProbFunc = MakeRBMTransProbFunc(ts[i+1], ts[i], c, d, mu, sigma, n_terms)

        if i == 0:
            densFunc = lambda x: transProbFunc(x, x0)
            legExpResult = LegExp(densFunc, maxDeg)
        else:
            legExpResult = LegExpSPDensity(legExpResultPrev.fApp, transProbFunc, maxDeg, epsabs=integEpsabs)

        coefs = np.zeros(maxDeg+1)
        coefs[0] = 0.5

        for l in range(1, maxDeg+1):
            a = 0.5 * (legExpResult.coefs[l] / (l + 0.5) + 1)
            rqaeResult = RQAE(a, eps, R)
            coefs[l] = (2 * rqaeResult.aEst -1) * (l + 0.5)
            totalQueryNum += rqaeResult.TotalQueryNum
            maxDepth = max(maxDepth, rqaeResult.MaxDepth)

        legExpResultPrev = LegExpResult(coefs)

    pEsts.append(sp.integrate.quad(legExpResultPrev.fApp, x0, 1.0)[0])
    totalQueryNums.append(totalQueryNum)
    maxDepths.append(maxDepth)
    NsComp.append(N)

    retDf = pd.DataFrame(dict(N=NsComp,
                              pTrue=np.repeat(pTrue, len(NsComp)),
                              eps=epss,
                              pEst=pEsts,
                              absErr=np.abs(np.array(pEsts) - pTrue),
                              totalQueryNum=totalQueryNums,
                              maxDepth=maxDepths))
    retDf.to_csv('DivideRBM_RQAE.csv', index=False)

2025-02-19 13:52:42.021786 N= 8
i_t= 0 2025-02-19 13:52:42.021786


c:\Users\koich\Desktop\Code\DivQCOSDE\QAE\MaximizeL.py:11: RuntimeWarning: divide by zero encountered in log
  neglogL = lambda theta: -np.dot(n1s, np.log(np.sin(thetaMuls * theta)**2)) - np.dot(n0s, np.log(np.cos(thetaMuls * theta)**2))


i_t= 1 2025-02-19 13:52:43.251813
i_t= 2 2025-02-19 13:52:50.074605
i_t= 3 2025-02-19 13:52:56.973104
i_t= 4 2025-02-19 13:53:03.856197
i_t= 5 2025-02-19 13:53:11.565615
i_t= 6 2025-02-19 13:53:19.459417
i_t= 7 2025-02-19 13:53:27.770707
2025-02-19 13:53:36.651528 N= 11
i_t= 0 2025-02-19 13:53:36.651528
i_t= 1 2025-02-19 13:53:38.057633
i_t= 2 2025-02-19 13:53:47.797642
i_t= 3 2025-02-19 13:53:57.704959
i_t= 4 2025-02-19 13:54:11.059359
i_t= 5 2025-02-19 13:54:22.311214
i_t= 6 2025-02-19 13:54:31.900556
i_t= 7 2025-02-19 13:54:43.936454
i_t= 8 2025-02-19 13:54:55.097609
i_t= 9 2025-02-19 13:55:04.300833
i_t= 10 2025-02-19 13:55:13.382356
2025-02-19 13:55:22.381087 N= 16
i_t= 0 2025-02-19 13:55:22.381087
i_t= 1 2025-02-19 13:55:23.757965
i_t= 2 2025-02-19 13:55:40.476321
i_t= 3 2025-02-19 13:55:57.254917
i_t= 4 2025-02-19 13:56:10.171976
i_t= 5 2025-02-19 13:56:23.071345
i_t= 6 2025-02-19 13:56:32.874922
i_t= 7 2025-02-19 13:56:42.802397
i_t= 8 2025-02-19 13:56:55.753755
i_t= 9 2025-02-

In [7]:
retDf

,N,pTrue,eps,pEst,absErr,totalQueryNum,maxDepth
0,8,0.649605,0.000345,0.649287,0.000318,2650827,2895
1,11,0.649605,0.000294,0.649912,0.000307,3817894,3395
2,16,0.649605,0.000244,0.649587,0.000018,5877573,4095
3,22,0.649605,0.000208,0.649674,0.000070,13964128,4802
4,32,0.649605,0.000173,0.649074,0.000530,21175491,5792
...,...,...,...,...,...,...,...
65,16,0.649605,0.000244,0.649524,0.000080,5912263,4095
66,22,0.649605,0.000208,0.649224,0.000380,13982467,4802
67,32,0.649605,0.000173,0.649818,0.000213,21227136,5792
68,45,0.649605,0.000146,0.649593,0.000012,31321290,6868


In [8]:
retDf.groupby('N')['absErr'].mean()

N
8     0.000228
11    0.000161
16    0.000155
22    0.000283
32    0.000275
45    0.000155
64    0.000329
Name: absErr, dtype: float64